# DISCOtoolkit quickstart (DISCO v1)

Find samples in the [DISCO](https://disco.bii.a-star.edu.sg/) database by metadata and cell type, download them, and search a gene's expression.
This notebook runs in Google Colab or any Python >= 3.9. It takes a few minutes the first time (the install pulls in scanpy).

## 1. Install

In [ ]:
!pip install -q -U discotoolkit

# or the latest development version from GitHub:
# !pip install -q "git+https://github.com/JinmiaoChenLab/DISCOtoolkit_py.git"

## 2. Check the install and the server

The toolkit talks to DISCO v1 by default. `dt.set_server("v2")` or `dt.set_server("https://.../disco_v3_api/")` switches server.

In [ ]:
import discotoolkit as dt

print("discotoolkit", dt.__version__)
print("server:", dt.get_server())

## 3. (Optional) Check that the server answers everything the toolkit needs

This is the same check the maintainers run. Every line should say PASS.

In [ ]:
!git clone -q --depth 1 https://github.com/JinmiaoChenLab/DISCOtoolkit_py.git /content/DISCOtoolkit_py
!python /content/DISCOtoolkit_py/tests/test_server_contract.py

## 4. Explore what is in DISCO

In [ ]:
print(sorted(map(str, dt.list_metadata_item("tissue")))[:15])      # tissues
print(dt.find_celltype("myoid"))                                    # search the cell ontology

## 5. Filter samples

Samples from control thymus (10x 3') that contain **thymic myoid cells** -- a rare cell type that is easy to miss if you only look at whole-tissue averages.

In [ ]:
flt = dt.Filter(
    tissue="thymus",
    platform=["10x3'"],
    sample_type=["control"],
    cell_type="Thymic myoid cell",
    cell_type_confidence="medium",     # "high", "medium" or "all"
    min_cell_per_sample=100,
)
metadata = dt.filter_disco_metadata(flt)
print(metadata.sample_count, "samples,", metadata.cell_count, "cells")
metadata.sample_metadata.head()

## 6. Download

Each sample is a 10x `.h5` converted to an AnnData `.h5ad` holding only the cells you filtered for.

In [ ]:
log = dt.download_disco_data(metadata, output_dir="disco_data")
!ls -lh disco_data

In [ ]:
import scanpy as sc
adata = sc.read("disco_data/" + metadata.sample_metadata["sample_id"].iloc[0] + ".h5ad")
adata

## 7. Search a gene

Expression of a gene across cell types and tissues, as on the DISCO website.

In [ ]:
dt.gene_search("ACTA1")

## More

- Cell type annotation with CELLiD: [CELLiD_celltype_annotation.ipynb](CELLiD_celltype_annotation.ipynb)
- Gene set enrichment: [scEnrichment.ipynb](scEnrichment.ipynb)
- Filtering and downloading in depth: [download_data.ipynb](download_data.ipynb)